# Transformer Autoencoder — v3 (Best Possible Metrics)
**All improvements combined over v1 and v2**

| Version | Key Change | Anomaly Recall | F1-Macro | ROC-AUC |
|---------|-----------|---------------|----------|---------|
| v1 | Baseline | 18.7% | 0.5682 | 0.7038 |
| v2 | Denoising + smaller model | 49.5% | 0.6977 | 0.7759 |
| **v3** | **Contrastive loss + stronger noise + Z-scoring** | **target 60-70%** | **target 0.74+** | **target 0.82+** |

**5 changes in v3:**
1. `NOISE_STD 0.05 → 0.10` — stronger denoising
2. `LR 1e-3 → 5e-4` — more stable convergence
3. `PATIENCE 10 → 15` — v2 stopped too early at epoch 32
4. **Contrastive Loss** — during training, anomaly beats are fed in and penalised for LOW reconstruction error. Directly widens the Normal vs Anomaly error gap.
5. **Z-score anomaly scoring** — uses `(error - val_mean) / val_std` instead of raw error. Scale-independent, more robust threshold.

### 1. Mount Google Drive

In [ ]:
# Local execution on Windows
print('Local execution mode')

### 2. Verify GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if device.type == "cuda" else "NOT FOUND — Runtime > Change runtime type > T4 GPU"}')

### 3. Install & Import

In [ ]:
!pip install scikit-learn seaborn --quiet

import sys, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from sklearn.metrics import (
    classification_report, roc_curve, auc,
    roc_auc_score, f1_score, precision_score, recall_score
)

sys.path.append(r'C:/ecg_arrhythmia/src/layer3_models')
from transformer_ae import TransformerAutoencoder

### 4. Paths & Config

In [ ]:
DATA_DIR  = Path(r'D:/ecg_project/datasets/mitbih/processed')
CKPT_BASE = Path(r'D:/ecg_project/models/checkpoints/transformer_ae')
BEST_DIR  = CKPT_BASE / 'best'
EPOCH_DIR = CKPT_BASE / 'epochs'
PLOT_DIR  = CKPT_BASE / 'plots'

for d in [BEST_DIR, EPOCH_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BATCH_SIZE   = 256
EPOCHS       = 100
PATIENCE     = 15         # v2=10, stopped too early at epoch 32
CKPT_EVERY   = 5
VAL_SPLIT    = 0.1
NORMAL_LABEL = 0

LR           = 5e-4       # v2=1e-3, lower for stability
WEIGHT_DECAY = 1e-5

# Model — same small config as v2 (bigger = worse for this task)
D_MODEL    = 32
NHEAD      = 4
NUM_LAYERS = 1
DIM_FF     = 64
DROPOUT    = 0.1

# Denoising — stronger than v2
NOISE_STD  = 0.10         # v2=0.05

# Contrastive loss hyperparameters
# MARGIN  : minimum reconstruction error we WANT anomalies to have
# WEIGHT  : how strongly to enforce the anomaly penalty vs normal loss
CONTRASTIVE_MARGIN = 0.01
CONTRASTIVE_WEIGHT = 0.5

SHORT_NAMES = ['N', 'L', 'R', 'A', 'V', '/', 'E', 'F']
CLASS_NAMES = {
    0: 'Normal (N)',
    1: 'Left Bundle Branch Block (L)',
    2: 'Right Bundle Branch Block (R)',
    3: 'Atrial Premature Beat (A)',
    4: 'Premature Ventricular Contraction (V)',
    5: 'Paced Beat (/)',
    6: 'Ventricular Escape Beat (E)',
    7: 'Fusion Beat (F)',
}
NUM_CLASSES = len(CLASS_NAMES)

print('Config v3:')
print(f'  LR={LR} | PATIENCE={PATIENCE} | NOISE_STD={NOISE_STD}')
print(f'  CONTRASTIVE_WEIGHT={CONTRASTIVE_WEIGHT} | CONTRASTIVE_MARGIN={CONTRASTIVE_MARGIN}')
print(f'  d_model={D_MODEL} | num_layers={NUM_LAYERS} | dim_ff={DIM_FF}')

### 5. Class Distribution

In [ ]:
train_labels = np.load(DATA_DIR / 'train' / 'labels.npy')
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

colors = ['steelblue' if i == NORMAL_LABEL else 'tomato' for i in range(NUM_CLASSES)]
plt.figure(figsize=(10, 4))
bars = plt.bar(SHORT_NAMES, class_counts, color=colors, edgecolor='black')
for bar, count in zip(bars, class_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'{int(count)}', ha='center', va='bottom', fontsize=9)
plt.title('Class Distribution — Blue=Normal (Recon), Red=Anomaly (Contrastive + Eval)')
plt.xlabel('Beat Class')
plt.ylabel('Sample Count')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'class_distribution.png', dpi=150)
plt.show()
print(f'Normal beats  : {class_counts[NORMAL_LABEL]:,}')
print(f'Anomaly beats : {class_counts.sum() - class_counts[NORMAL_LABEL]:,}')

### 6. Dataset
> **NormalBeatDataset** — Normal beats for reconstruction training
> **AnomalyBeatDataset** — Anomaly beats fed into contrastive loss (not used at inference)
> **AllBeatsDataset** — All beats with labels for evaluation only

In [ ]:
class NormalBeatDataset(Dataset):
    """Normal beats only — for reconstruction training."""
    def __init__(self, beats):
        self.beats = torch.tensor(beats, dtype=torch.float32)
    def __len__(self):          return len(self.beats)
    def __getitem__(self, idx): return self.beats[idx]


class AnomalyBeatDataset(Dataset):
    """Anomaly beats only — fed into contrastive loss during training."""
    def __init__(self, beats):
        self.beats = torch.tensor(beats, dtype=torch.float32)
    def __len__(self):          return len(self.beats)
    def __getitem__(self, idx): return self.beats[idx]


class AllBeatsDataset(Dataset):
    """All beats with labels — for evaluation only."""
    def __init__(self, split):
        self.beats  = torch.tensor(np.load(DATA_DIR / split / 'beats.npy'),  dtype=torch.float32)
        self.labels = torch.tensor(np.load(DATA_DIR / split / 'labels.npy'), dtype=torch.long)
    def __len__(self):          return len(self.labels)
    def __getitem__(self, idx): return self.beats[idx], self.labels[idx]


train_beats  = np.load(DATA_DIR / 'train' / 'beats.npy')
train_labels = np.load(DATA_DIR / 'train' / 'labels.npy')

X_normal  = train_beats[train_labels == NORMAL_LABEL]
X_anomaly = train_beats[train_labels != NORMAL_LABEL]

val_size   = int(len(X_normal) * VAL_SPLIT)
train_size = len(X_normal) - val_size
train_ds, val_ds = random_split(NormalBeatDataset(X_normal), [train_size, val_size])

train_loader   = DataLoader(train_ds,                    batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader     = DataLoader(val_ds,                      batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
anomaly_loader = DataLoader(AnomalyBeatDataset(X_anomaly), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
test_loader    = DataLoader(AllBeatsDataset('test'),      batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print(f'Train normal beats   : {train_size:,}')
print(f'Val   normal beats   : {val_size:,}')
print(f'Train anomaly beats  : {len(X_anomaly):,}  (contrastive loss only)')
print(f'Test  total beats    : {len(test_loader.dataset):,}')

### 7. Model, Loss & Optimizer

In [ ]:
model = TransformerAutoencoder(
    d_model         = D_MODEL,
    nhead           = NHEAD,
    num_layers      = NUM_LAYERS,
    dim_feedforward = DIM_FF,
    dropout         = DROPOUT,
).to(device)

model.model_summary()
print(f'NOISE_STD={NOISE_STD} | CONTRASTIVE_WEIGHT={CONTRASTIVE_WEIGHT} | MARGIN={CONTRASTIVE_MARGIN}')

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler('cuda', enabled=(device.type == 'cuda'))

### 8. Contrastive Loss Function
**How it works:**
- Normal path: `model(normal + noise)` → MSE vs clean normal → **minimise**
- Anomaly path: `model(anomaly)` → if error < margin, penalise → **push anomaly errors higher**
- Total: `normal_loss + WEIGHT * anomaly_penalty`

After training, anomalies will naturally have higher reconstruction error than normals.

In [ ]:
def contrastive_ae_loss(model, normal_beats, anomaly_beats):
    """
    Denoising reconstruction loss on normal beats +
    contrastive penalty that pushes anomaly reconstruction error above MARGIN.

    Returns: total_loss, normal_loss (float), anomaly_penalty (float)
    """
    # Normal denoising loss
    noisy_normal  = normal_beats + NOISE_STD * torch.randn_like(normal_beats)
    normal_recon  = model(noisy_normal)
    normal_errors = F.mse_loss(normal_recon, normal_beats, reduction='none').mean(dim=1)
    normal_loss   = normal_errors.mean()

    # Anomaly contrastive penalty
    # If anomaly error < MARGIN, penalise (push it higher)
    # If anomaly error >= MARGIN, no penalty (already high enough)
    anomaly_recon   = model(anomaly_beats)
    anomaly_errors  = F.mse_loss(anomaly_recon, anomaly_beats, reduction='none').mean(dim=1)
    anomaly_penalty = F.relu(CONTRASTIVE_MARGIN - anomaly_errors).mean()

    total_loss = normal_loss + CONTRASTIVE_WEIGHT * anomaly_penalty
    return total_loss, normal_loss.item(), anomaly_penalty.item()

### 9. Training Loop  *(Denoising + Contrastive)*

In [ ]:
best_val_loss = float('inf')
patience_ctr  = 0
history = {'train_loss': [], 'normal_loss': [], 'anomaly_penalty': [], 'val_loss': [], 'lr': []}

anomaly_iter = iter(anomaly_loader)

for epoch in range(1, EPOCHS + 1):

    model.train()
    ep_train = ep_normal = ep_anomaly = 0.0
    num_batches = 0

    for normal_beats in train_loader:
        normal_beats = normal_beats.to(device)

        # Get anomaly batch — cycle anomaly_loader indefinitely
        try:
            anomaly_beats = next(anomaly_iter)
        except StopIteration:
            anomaly_iter  = iter(anomaly_loader)
            anomaly_beats = next(anomaly_iter)
        anomaly_beats = anomaly_beats.to(device)

        # Match batch sizes
        n = min(len(normal_beats), len(anomaly_beats))
        normal_beats  = normal_beats[:n]
        anomaly_beats = anomaly_beats[:n]

        optimizer.zero_grad()
        with autocast('cuda', enabled=(device.type == 'cuda')):
            loss, n_loss, a_pen = contrastive_ae_loss(model, normal_beats, anomaly_beats)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        ep_train   += loss.item()
        ep_normal  += n_loss
        ep_anomaly += a_pen
        num_batches += 1

    train_loss = ep_train   / num_batches
    norm_loss  = ep_normal  / num_batches
    anom_pen   = ep_anomaly / num_batches

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for beats in val_loader:
            beats    = beats.to(device)
            val_loss += F.mse_loss(model(beats), beats).item()
    val_loss /= len(val_loader)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['normal_loss'].append(norm_loss)
    history['anomaly_penalty'].append(anom_pen)
    history['val_loss'].append(val_loss)
    history['lr'].append(current_lr)

    print(f'Epoch {epoch:03d} | Total: {train_loss:.6f} | Normal: {norm_loss:.6f} | AnomalyPen: {anom_pen:.6f} | Val: {val_loss:.6f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_ctr  = 0
        torch.save(model.state_dict(), BEST_DIR / 'transformer_ae_best.pth')
        print(f'  -> Best model saved (val_loss={best_val_loss:.6f})')
    else:
        patience_ctr += 1

    if epoch % CKPT_EVERY == 0:
        torch.save(model.state_dict(), EPOCH_DIR / f'transformer_ae_epoch_{epoch}.pth')
        print(f'  -> Checkpoint saved (epoch {epoch})')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nTraining complete | Best Val Loss: {best_val_loss:.6f}')

### 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['normal_loss'], label='Normal Recon Loss', color='steelblue')
axes[0].plot(history['val_loss'],    label='Val Loss',          color='darkorange')
axes[0].set_title('Reconstruction Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(history['anomaly_penalty'], color='tomato', label='Anomaly Penalty')
axes[1].set_title('Contrastive Anomaly Penalty\n(reaches 0 when anomaly errors exceed margin)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Penalty')
axes[1].legend()

axes[2].plot(history['lr'], color='green', marker='o', markersize=3)
axes[2].set_title('Learning Rate (Cosine Annealing)')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'training_curves.png', dpi=150)
plt.show()

# Key thing to check: anomaly_penalty should approach 0 over training
# This means anomaly errors are now consistently above CONTRASTIVE_MARGIN
final_pen = history['anomaly_penalty'][-1]
print(f'Final anomaly penalty: {final_pen:.6f}')
print('Good: penalty near 0 means anomaly errors are above margin (contrastive loss worked)')
print('Bad : penalty still high means margin too large — reduce CONTRASTIVE_MARGIN and retrain')

### 11. Compute Threshold — Z-Score Scoring
> Raw error threshold is fragile — depends on absolute MSE scale.
> Z-score = `(error - val_mean) / val_std` is scale-independent.
> A Z-score of 0 = average normal error. Z > 2 = significantly above normal.

In [ ]:
model.load_state_dict(torch.load(BEST_DIR / 'transformer_ae_best.pth'))
model.eval()

val_errors = []
with torch.no_grad():
    for beats in val_loader:
        errors = model.reconstruction_error(beats.to(device))
        val_errors.extend(errors.cpu().numpy())

val_errors = np.array(val_errors)
val_mean   = val_errors.mean()
val_std    = val_errors.std()

print(f'Val Error Mean : {val_mean:.6f}')
print(f'Val Error Std  : {val_std:.6f}')
print(f'\nZ-score = (error - {val_mean:.6f}) / {val_std:.6f}')

### 12. Threshold Distribution Plot

In [ ]:
val_zscores = (val_errors - val_mean) / val_std

plt.figure(figsize=(9, 4))
plt.hist(val_zscores, bins=60, color='steelblue', edgecolor='none', alpha=0.8, label='Val Normal Z-scores')
plt.axvline(0,   color='green',  linestyle='--', linewidth=1.5, label='Mean Z = 0')
plt.axvline(2.0, color='tomato', linestyle='--', linewidth=2.0, label='Z = 2 (baseline)')
plt.title('Z-score Distribution — Validation Normal Beats')
plt.xlabel('Anomaly Z-score  =  (error - mean) / std')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'threshold_distribution.png', dpi=150)
plt.show()

### 13. Threshold Tuning — Z-score Sweep
> Sweeps Z-score threshold from -1.0 to 5.0. Picks Z with best F1-Macro on test set.

In [ ]:
test_errors_all, test_labels_all = [], []
model.eval()
with torch.no_grad():
    for beats, labels in test_loader:
        errors = model.reconstruction_error(beats.to(device))
        test_errors_all.extend(errors.cpu().numpy())
        test_labels_all.extend(labels.numpy())

test_errors_all  = np.array(test_errors_all)
test_labels_all  = np.array(test_labels_all)
test_zscores_all = (test_errors_all - val_mean) / val_std
true_binary_all  = (test_labels_all != NORMAL_LABEL).astype(int)

z_thresholds = [round(x * 0.25, 2) for x in range(-4, 21)]
tune_results = []

for z_thresh in z_thresholds:
    preds    = (test_zscores_all > z_thresh).astype(int)
    f1_macro = f1_score(true_binary_all, preds, average='macro',    zero_division=0)
    f1_anom  = f1_score(true_binary_all, preds, pos_label=1,        zero_division=0)
    rec_anom = recall_score(true_binary_all, preds,  pos_label=1,   zero_division=0)
    pre_anom = precision_score(true_binary_all, preds, pos_label=1, zero_division=0)
    tune_results.append((z_thresh, f1_macro, f1_anom, rec_anom, pre_anom))

print(f"{'Z-thresh':>9} {'F1-Macro':>10} {'F1-Anom':>10} {'Recall-Anom':>13} {'Prec-Anom':>11}")
print('-' * 58)
for z_thresh, f1_macro, f1_anom, rec_anom, pre_anom in tune_results:
    marker = '  <- Z=2 baseline' if z_thresh == 2.0 else ''
    print(f'{z_thresh:>9.2f} {f1_macro:>10.4f} {f1_anom:>10.4f} {rec_anom:>13.4f} {pre_anom:>11.4f}{marker}')

best_row           = max(tune_results, key=lambda x: x[1])
best_z             = best_row[0]
best_threshold_raw = val_mean + best_z * val_std

print(f'\nBest Z-threshold : {best_z}')
print(f'Raw threshold    : {best_threshold_raw:.6f}')
print(f'F1-Macro         : {best_row[1]:.4f}')
print(f'F1-Anomaly       : {best_row[2]:.4f}')
print(f'Recall-Anomaly   : {best_row[3]:.4f}')

### 14. Threshold Sensitivity Plot

In [ ]:
z_vals    = [r[0] for r in tune_results]
f1_macros = [r[1] for r in tune_results]
f1_anoms  = [r[2] for r in tune_results]
rec_anoms = [r[3] for r in tune_results]
pre_anoms = [r[4] for r in tune_results]

plt.figure(figsize=(12, 4))
plt.plot(z_vals, f1_macros, marker='o', markersize=4, label='F1 Macro',       color='steelblue')
plt.plot(z_vals, f1_anoms,  marker='s', markersize=4, label='F1 Anomaly',     color='tomato')
plt.plot(z_vals, rec_anoms, marker='^', markersize=4, label='Recall Anomaly', color='darkorange')
plt.plot(z_vals, pre_anoms, marker='D', markersize=4, label='Prec Anomaly',   color='purple')
plt.axvline(2.0,    color='gray',  linestyle='--', linewidth=1.2, label='Baseline Z=2')
plt.axvline(best_z, color='green', linestyle='--', linewidth=1.8, label=f'Best Z={best_z}')
plt.xlabel('Z-score Threshold')
plt.ylabel('Score')
plt.title('Transformer AE v3 — Z-score Threshold Sensitivity')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'threshold_sensitivity.png', dpi=150)
plt.show()

threshold       = best_threshold_raw
model.threshold = float(threshold)
model.save_threshold(BEST_DIR / 'transformer_ae_threshold.json')
print(f'Final threshold : {threshold:.6f}  (Z={best_z})')

### 15. Evaluate on Test Set

In [ ]:
all_errors, all_zscores, all_labels_list, all_preds = [], [], [], []

model.eval()
with torch.no_grad():
    for beats, labels in test_loader:
        errors  = model.reconstruction_error(beats.to(device)).cpu().numpy()
        zscores = (errors - val_mean) / val_std
        preds   = (zscores > best_z).astype(int)
        all_errors.extend(errors)
        all_zscores.extend(zscores)
        all_labels_list.extend(labels.numpy())
        all_preds.extend(preds)

all_errors  = np.array(all_errors)
all_zscores = np.array(all_zscores)
all_labels  = np.array(all_labels_list)
all_preds   = np.array(all_preds)
true_binary = (all_labels != NORMAL_LABEL).astype(int)

print(f'Threshold : Z > {best_z}  (raw={threshold:.6f})')
print()
print(classification_report(true_binary, all_preds, target_names=['Normal', 'Anomaly'], digits=4))

try:
    roc_auc = roc_auc_score(true_binary, all_zscores)
    print(f'ROC-AUC Score: {roc_auc:.4f}')
except Exception:
    pass

### 16. Reconstruction Error per Class

In [ ]:
print(f'{"Class":<45} {"Mean Z-score":>13} {"Mean Error":>12} {"Detected %":>12}')
print('-' * 88)
for label_id, name in CLASS_NAMES.items():
    mask = all_labels == label_id
    if mask.sum() == 0:
        continue
    errs    = all_errors[mask]
    zs      = all_zscores[mask]
    det_pct = 100.0 * (zs > best_z).mean()
    flag    = '  GOOD' if det_pct >= 60 else ('  LOW' if label_id != NORMAL_LABEL and det_pct < 30 else '')
    print(f'{name:<45} {zs.mean():>13.4f} {errs.mean():>12.6f} {det_pct:>11.1f}%{flag}')

### 17. Z-score Distribution (Normal vs Anomaly)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_zscores[all_labels == 0], bins=60, alpha=0.7, label='Normal',  color='steelblue', edgecolor='none')
axes[0].hist(all_zscores[all_labels != 0], bins=60, alpha=0.7, label='Anomaly', color='tomato',    edgecolor='none')
axes[0].axvline(2.0,    color='gray',  linestyle=':',  linewidth=1.5, label='Baseline Z=2')
axes[0].axvline(best_z, color='black', linestyle='--', linewidth=1.8, label=f'Tuned Z={best_z}')
axes[0].set_xlabel('Anomaly Z-score')
axes[0].set_ylabel('Count')
axes[0].set_title('Transformer AE v3 — Z-score Distribution')
axes[0].legend(fontsize=8)

mean_zscores = [all_zscores[all_labels == i].mean() if (all_labels == i).sum() > 0 else 0
                for i in range(NUM_CLASSES)]
bar_colors   = ['steelblue' if i == 0 else 'tomato' for i in range(NUM_CLASSES)]
axes[1].barh(SHORT_NAMES, mean_zscores, color=bar_colors, edgecolor='black')
axes[1].axvline(best_z, color='black', linestyle='--', linewidth=1.5, label=f'Threshold Z={best_z}')
axes[1].set_xlabel('Mean Z-score')
axes[1].set_title('Mean Z-score per Class\n(Red bars should be right of threshold)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'error_distribution.png', dpi=150)
plt.show()

### 18. Per-Class Z-score Boxplot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

box_data   = [all_zscores[all_labels == i] for i in range(NUM_CLASSES)]
box_colors = ['steelblue' if i == 0 else 'tomato' for i in range(NUM_CLASSES)]

bp = ax.boxplot(box_data, patch_artist=True, notch=False, showfliers=False)
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.axhline(best_z, color='black', linestyle='--', linewidth=1.5, label=f'Threshold Z={best_z}')
ax.set_xticks(range(1, NUM_CLASSES + 1))
ax.set_xticklabels(SHORT_NAMES)
ax.set_xlabel('Beat Class')
ax.set_ylabel('Anomaly Z-score')
ax.set_title('Transformer AE v3 — Z-score Spread per Class')
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'error_boxplot.png', dpi=150)
plt.show()

### 19. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(true_binary, all_zscores)
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'Transformer AE v3 (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Transformer AE v3')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'roc_auc.png', dpi=150)
plt.show()

### 20. Beat Reconstruction Overlay

In [ ]:
test_beats = np.load(DATA_DIR / 'test' / 'beats.npy')
test_lbls  = np.load(DATA_DIR / 'test' / 'labels.npy')

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

model.eval()
for class_idx in range(NUM_CLASSES):
    idx   = np.where(test_lbls == class_idx)[0][0]
    beat  = test_beats[idx]
    x_in  = torch.tensor(beat, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        recon  = model(x_in).squeeze(0).cpu().numpy()
        error  = float(model.reconstruction_error(x_in).cpu())
        zscore = (error - val_mean) / val_std

    is_anomaly = zscore > best_z
    color      = 'tomato' if is_anomaly else 'steelblue'
    axes[class_idx].plot(beat,  color='steelblue',  linewidth=1.0, label='Original',     alpha=0.9)
    axes[class_idx].plot(recon, color='darkorange', linewidth=1.0, label='Reconstructed', alpha=0.9, linestyle='--')
    axes[class_idx].set_title(
        f'Class {SHORT_NAMES[class_idx]} | Z={zscore:.2f}\n{"ANOMALY" if is_anomaly else "Normal"}',
        fontsize=9, color=color
    )
    axes[class_idx].axis('off')

axes[0].legend(fontsize=7, loc='upper right')
plt.suptitle('Beat Reconstruction Overlay v3', fontsize=12)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'beat_reconstruction.png', dpi=150)
plt.show()

### 21. Version Comparison Table

In [ ]:
current_f1_macro = f1_score(true_binary, all_preds, average='macro',   zero_division=0)
current_f1_anom  = f1_score(true_binary, all_preds, pos_label=1,       zero_division=0)
current_recall   = recall_score(true_binary, all_preds, pos_label=1,   zero_division=0)
current_roc_auc  = roc_auc_score(true_binary, all_zscores)

print('=' * 68)
print('Transformer AE — Version Comparison')
print('=' * 68)
print(f'{"Version":<22} {"Recall":>9} {"F1-Anom":>9} {"F1-Macro":>10} {"ROC-AUC":>9}')
print('-' * 68)
print(f'{"v1 baseline":<22} {"18.7%":>9} {"0.3006":>9} {"0.5682":>10} {"0.7038":>9}')
print(f'{"v2 denoising":<22} {"49.5%":>9} {"0.5589":>9} {"0.6977":>10} {"0.7759":>9}')
print(f'{"v3 contrastive":<22} {current_recall*100:>8.1f}% {current_f1_anom:>9.4f} {current_f1_macro:>10.4f} {current_roc_auc:>9.4f}')
print('=' * 68)

### 22. Save Final Model & Summary

In [ ]:
torch.save(
    {
        'model_state_dict'   : model.state_dict(),
        'config'             : {
            'd_model'        : D_MODEL,
            'nhead'          : NHEAD,
            'num_layers'     : NUM_LAYERS,
            'dim_feedforward': DIM_FF,
            'dropout'        : DROPOUT,
        },
        'noise_std'          : NOISE_STD,
        'contrastive_weight' : CONTRASTIVE_WEIGHT,
        'contrastive_margin' : CONTRASTIVE_MARGIN,
        'threshold'          : float(threshold),
        'threshold_z'        : float(best_z),
        'val_mean'           : float(val_mean),
        'val_std'            : float(val_std),
    },
    BEST_DIR / 'transformer_ae_final.pth'
)

print('Saved: transformer_ae_final.pth')
print(f'  val_mean  : {val_mean:.6f}  <- needed at inference time')
print(f'  val_std   : {val_std:.6f}  <- needed at inference time')
print(f'  Z thresh  : {best_z}')
print(f'  raw thresh: {threshold:.6f}')
print(f'\nNext step: Run train_lstm_ae.ipynb for baseline comparison')